# 第2章: 分類のためのシンプルな機械学習アルゴリズムの学習

この Notebook は、原本 `machine-learning-book/ch02/ch02.ipynb` を最新の Python / NumPy / Matplotlib 環境で継続検証できる形に移行したものです。
原本の教育的な流れを保ちながら、外部 URL 依存、Notebook マジック、古い NumPy API、CI に不要な保存処理を取り除いています。


## この Notebook で確認すること

- 現在の `uv` 環境で Python と主要パッケージのバージョンを確認する。
- 読み取り専用の原本リポジトリから図版と `iris.data` を安全に参照する。
- Iris データセットを使って `Perceptron`、`AdalineGD`、`AdalineSGD` を最新環境で再現する。
- `pytest --nbmake` によるヘッドレス実行でも停止しないことを確認する。


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
from IPython.display import Image, display

PACKAGE_NAMES = ["numpy", "pandas", "matplotlib", "scikit-learn"]
from sklearn.datasets import load_iris

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / '.github/workflows/ci.yml').exists() and (candidate / 'machine-learning-book').exists():
            return candidate
    raise FileNotFoundError('リポジトリルートを特定できませんでした')

REPO_ROOT = find_repo_root(Path.cwd())
CHAPTER_DIR = REPO_ROOT / 'machine-learning-book' / 'ch02'
FIG_DIR = CHAPTER_DIR / 'figures'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'

plt.style.use('seaborn-v0_8-whitegrid')

print(f'Python 実行ファイル: {sys.executable}')
print(f'Python バージョン: {platform.python_version()}')
print(f'Matplotlib バックエンド: {matplotlib.get_backend()}')
print(f'原本 ch02 ディレクトリ: {CHAPTER_DIR}')


In [ ]:
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)
package_versions


## 原本図版の参照

原本の説明で使われている図版アセットは、読み取り専用の `machine-learning-book/ch02/figures/` に残します。
移行版 Notebook では、実装コードをルート側に置いたまま相対パスに依存しない探索でこれらを再利用します。


In [ ]:
selected_figures = [
    ('02_01.png', 520),
    ('02_02.png', 520),
    ('02_03.png', 620),
    ('02_04.png', 620),
    ('02_09.png', 620),
    ('02_10.png', 520),
]

for name, width in selected_figures:
    figure_path = FIG_DIR / name
    print(name)
    display(Image(filename=str(figure_path), width=width))


## Iris データの読み込み

`seaborn.load_dataset('iris')` でも同様の表は取得できますが、実際にはオンライン上のデータ取得に依存します。
CI の再現性を優先し、この移行版では `scikit-learn` に同梱されている `load_iris(as_frame=True)` を使ってローカルに完結させます。
章の前半に合わせて、`Setosa` と `Versicolor` の 2 クラス、`sepal length` と `petal length` の 2 特徴量だけを使います。


In [ ]:
iris_bunch = load_iris(as_frame=True)
df = iris_bunch.frame.copy()
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'target']
df['class_label'] = df['target'].map(dict(enumerate(iris_bunch.target_names)))

y = df.iloc[:100]['target'].to_numpy()
X = df.iloc[:100][['sepal_length', 'petal_length']].to_numpy(dtype=float)

df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X[:50, 0], X[:50, 1], color='tab:red', marker='o', label='Setosa')
ax.scatter(X[50:100, 0], X[50:100, 1], color='tab:blue', marker='s', label='Versicolor')
ax.set_xlabel('Sepal length [cm]')
ax.set_ylabel('Petal length [cm]')
ax.set_title('Iris データの 2 クラス分布')
ax.legend(loc='upper left')
plt.show()
plt.close(fig)


## Perceptron の実装と学習

原本のオブジェクト指向 API を維持しつつ、NumPy 2 系でも安定して動くように `np.float_` を通常の `float` に置き換えています。
学習の更新回数を追跡し、分類境界も可視化します。


In [ ]:
class Perceptron:
    def __init__(self, eta: float = 0.01, n_iter: int = 50, random_state: int = 1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    def fit(self, X: np.ndarray, y: np.ndarray):
        rgen = np.random.RandomState(self.random_state)
        self.w_ = rgen.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b_ = 0.0
        self.errors_ = []

        for _ in range(self.n_iter):
            errors = 0
            for xi, target in zip(X, y):
                update = self.eta * (target - self.predict(xi))
                self.w_ += update * xi
                self.b_ += update
                errors += int(update != 0.0)
            self.errors_.append(errors)
        return self

    def net_input(self, X: np.ndarray) -> np.ndarray:
        return np.dot(X, self.w_) + self.b_

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.where(self.net_input(X) >= 0.0, 1, 0)

v1 = np.array([1.0, 2.0, 3.0])
v2 = 0.5 * v1
angle = np.degrees(np.arccos(v1.dot(v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))))
print(f'同方向ベクトルの角度: {angle:.1f} 度')


In [ ]:
def plot_decision_regions(X: np.ndarray, y: np.ndarray, classifier, resolution: float = 0.02):
    markers = ('o', 's', '^', 'v', '<')
    colors = ('tab:red', 'tab:blue', 'lightgreen', 'gray', 'cyan')
    cmap = ListedColormap(colors[: len(np.unique(y))])

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(
        np.arange(x1_min, x1_max, resolution),
        np.arange(x2_min, x2_max, resolution),
    )
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    lab = classifier.predict(grid).reshape(xx1.shape)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.contourf(xx1, xx2, lab, alpha=0.3, cmap=cmap)
    ax.set_xlim(xx1.min(), xx1.max())
    ax.set_ylim(xx2.min(), xx2.max())

    for idx, cl in enumerate(np.unique(y)):
        ax.scatter(
            x=X[y == cl, 0],
            y=X[y == cl, 1],
            alpha=0.85,
            c=colors[idx],
            marker=markers[idx],
            label=f'Class {cl}',
            edgecolor='black',
        )
    return fig, ax


In [ ]:
ppn = Perceptron(eta=0.1, n_iter=10).fit(X, y)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(ppn.errors_) + 1), ppn.errors_, marker='o')
ax.set_xlabel('Epochs')
ax.set_ylabel('Number of updates')
ax.set_title('Perceptron の学習更新回数')
plt.show()
plt.close(fig)

fig, ax = plot_decision_regions(X, y, classifier=ppn)
ax.set_xlabel('Sepal length [cm]')
ax.set_ylabel('Petal length [cm]')
ax.set_title('Perceptron の決定領域')
ax.legend(loc='upper left')
plt.show()
plt.close(fig)


## Adaline と勾配降下法

続いて、原本と同様に二乗誤差を最小化する `AdalineGD` を実装します。
学習率による収束の違いを確認した後、特徴量標準化によって学習が安定することを確かめます。


In [ ]:
class AdalineGD:
    def __init__(self, eta: float = 0.01, n_iter: int = 50, random_state: int = 1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    def fit(self, X: np.ndarray, y: np.ndarray):
        rgen = np.random.RandomState(self.random_state)
        self.w_ = rgen.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b_ = 0.0
        self.losses_ = []

        for _ in range(self.n_iter):
            output = self.activation(self.net_input(X))
            errors = y - output
            self.w_ += self.eta * 2.0 * X.T.dot(errors) / X.shape[0]
            self.b_ += self.eta * 2.0 * errors.mean()
            self.losses_.append((errors ** 2).mean())
        return self

    def net_input(self, X: np.ndarray) -> np.ndarray:
        return np.dot(X, self.w_) + self.b_

    def activation(self, X: np.ndarray) -> np.ndarray:
        return X

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.where(self.activation(self.net_input(X)) >= 0.5, 1, 0)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))

ada_fast = AdalineGD(n_iter=15, eta=0.1).fit(X, y)
axes[0].plot(range(1, len(ada_fast.losses_) + 1), np.log10(ada_fast.losses_), marker='o')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('log10(Mean squared error)')
axes[0].set_title('学習率 0.1')

ada_slow = AdalineGD(n_iter=15, eta=0.0001).fit(X, y)
axes[1].plot(range(1, len(ada_slow.losses_) + 1), ada_slow.losses_, marker='o')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Mean squared error')
axes[1].set_title('学習率 0.0001')

fig.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
X_std = X.copy()
X_std[:, 0] = (X[:, 0] - X[:, 0].mean()) / X[:, 0].std()
X_std[:, 1] = (X[:, 1] - X[:, 1].mean()) / X[:, 1].std()

ada_gd = AdalineGD(n_iter=20, eta=0.5).fit(X_std, y)

fig, ax = plot_decision_regions(X_std, y, classifier=ada_gd)
ax.set_title('Adaline - Gradient descent')
ax.set_xlabel('Sepal length [standardized]')
ax.set_ylabel('Petal length [standardized]')
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(ada_gd.losses_) + 1), ada_gd.losses_, marker='o')
ax.set_xlabel('Epochs')
ax.set_ylabel('Mean squared error')
ax.set_title('標準化後の AdalineGD の収束')
fig.tight_layout()
plt.show()
plt.close(fig)


## 確率的勾配降下法版 Adaline

章の最後では、各サンプルごとに重みを更新する `AdalineSGD` を確認します。
これも原本の流れを維持しつつ、ファイル保存処理を省いた CI 向けの構成にしています。


In [ ]:
class AdalineSGD:
    def __init__(self, eta: float = 0.01, n_iter: int = 10, shuffle: bool = True, random_state: int | None = None):
        self.eta = eta
        self.n_iter = n_iter
        self.shuffle = shuffle
        self.random_state = random_state
        self.w_initialized = False

    def fit(self, X: np.ndarray, y: np.ndarray):
        self._initialize_weights(X.shape[1])
        self.losses_ = []
        for _ in range(self.n_iter):
            if self.shuffle:
                X, y = self._shuffle(X, y)
            losses = [self._update_weights(xi, target) for xi, target in zip(X, y)]
            self.losses_.append(float(np.mean(losses)))
        return self

    def partial_fit(self, X: np.ndarray, y: np.ndarray):
        X = np.atleast_2d(X)
        y = np.atleast_1d(y)
        if not self.w_initialized:
            self._initialize_weights(X.shape[1])
        for xi, target in zip(X, y):
            self._update_weights(xi, target)
        return self

    def _shuffle(self, X: np.ndarray, y: np.ndarray):
        r = self.rgen.permutation(len(y))
        return X[r], y[r]

    def _initialize_weights(self, m: int):
        self.rgen = np.random.RandomState(self.random_state)
        self.w_ = self.rgen.normal(loc=0.0, scale=0.01, size=m)
        self.b_ = 0.0
        self.w_initialized = True

    def _update_weights(self, xi: np.ndarray, target: float) -> float:
        output = self.activation(self.net_input(xi))
        error = target - output
        self.w_ += self.eta * 2.0 * xi * error
        self.b_ += self.eta * 2.0 * error
        return float(error ** 2)

    def net_input(self, X: np.ndarray) -> np.ndarray:
        return np.dot(X, self.w_) + self.b_

    def activation(self, X: np.ndarray) -> np.ndarray:
        return X

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.where(self.activation(self.net_input(X)) >= 0.5, 1, 0)

ada_sgd = AdalineSGD(n_iter=15, eta=0.01, random_state=1).fit(X_std, y)

fig, ax = plot_decision_regions(X_std, y, classifier=ada_sgd)
ax.set_title('Adaline - Stochastic gradient descent')
ax.set_xlabel('Sepal length [standardized]')
ax.set_ylabel('Petal length [standardized]')
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(ada_sgd.losses_) + 1), ada_sgd.losses_, marker='o')
ax.set_xlabel('Epochs')
ax.set_ylabel('Average loss')
ax.set_title('AdalineSGD の平均損失')
fig.tight_layout()
plt.show()
plt.close(fig)

ada_sgd.partial_fit(X_std[0, :], y[0])
print('partial_fit を 1 サンプルで実行できました')


## まとめ

第2章の移行版では、原本の主要な学習アルゴリズム実装と可視化を維持しながら、以下を現行環境向けに更新しました。

- `iris.data` へのファイル依存をやめ、`scikit-learn` 同梱の Iris データセットを使う。
- Notebook マジックや図の保存セルを除去し、`nbmake` のヘッドレス実行で止まらない形にする。
- NumPy 2 系で非推奨または互換性リスクのある書き方を避ける。
- `src/` 配下からでも原本アセットを解決できるようにする。
